# Notebook 02 — Cascade and Simulation

**Purpose:** Define the ten-parameter cascade of spatial migration feasibility, quantify its uncertainty through an unconditional variance decomposition, and compute conditional Monte Carlo commitment depths that condition on observed per-hour grid states. Produces the headline simulation results cited in NE manuscript §2 and §5.

**Inputs** (from notebook 01):
- `outputs/contracts/stress_correlation_results.json`
- `outputs/contracts/per_hour_destination_availability.parquet`
- `outputs/contracts/workload_parameters.json`

**Outputs** (consumed by notebook 03):
- `outputs/contracts/cascade_parameters.json`
- `outputs/contracts/conditional_mc_results.json`
- `outputs/tables/sensitivity_surface.csv`
- `outputs/tables/sensitivity_tornado.csv`

---

## Notebook Architecture

| Part | Section | Contents |
|---|---|---|
| **0** | Setup | Imports, REPO_ROOT, contract loading, sanity checks |
| **1** | Cascade Framework (§2) | 10-parameter definitions, ranges, static product, commitment depth |
| **2** | Unconditional Variance Decomposition (§2, Methods) | MC over all parameters, η² attribution, scenario summary |
| **3** | Conditional Monte Carlo (§5) | Single facility, empirical fleet, per-GW sweep |
| **4** | Sensitivity Analysis (§5, Methods) | 2D surface, tornado |
| **5** | Contract Exports | JSONs + sensitivity CSVs |

## Headline Results (reconciled parameter set)

| Claim | Value | Cell |
|---|---|---|
| Cascade central product | ~0.0384 | 1-3 |
| Commitment depth (cascade baseline) | ~20.4% | 1-4 |
| Variance shares D2/D4/D5/S2 | 44/28/13/13% (verify) | 2-1 |
| Single-facility MC commitment depth | TBD from regression | 3-2 |
| Empirical fleet MC commitment depth | TBD from regression | 3-3 |
| Per-GW sweep @ 10 GW | TBD from regression | 3-4 |

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-1: IMPORTS, PATH RESOLUTION, CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats

# ─── REPO_ROOT resolver ──────────────────────────────────────────────────────
# Kept in sync with notebooks/01_empirical_evidence.ipynb Cell 2-1. Resolves
# the repo root whether the notebook is launched from the repo root (VS Code,
# Jupyter started at repo root) or from notebooks/ (nbconvert default, or
# Jupyter started inside notebooks/).
# ─────────────────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
if (_cwd / "notebooks").exists() and (_cwd / "data").exists():
    REPO_ROOT = _cwd                      # launched from repo root
elif _cwd.name == "notebooks":
    REPO_ROOT = _cwd.parent               # launched from notebooks/
else:
    REPO_ROOT = _cwd                      # fallback

# Guard against silent mis-resolution: if REPO_ROOT doesn't look like the repo,
# fail loudly here rather than further downstream with a confusing path error.
assert (REPO_ROOT / "notebooks").exists() and (REPO_ROOT / "outputs").exists(), (
    f"REPO_ROOT resolved to {REPO_ROOT}, which does not look like the repo root "
    f"(missing notebooks/ or outputs/). Launch Jupyter from the repo root or "
    f"from notebooks/."
)

# ─── Derived paths ───────────────────────────────────────────────────────────
CONTRACTS_DIR = REPO_ROOT / "outputs" / "contracts"
FIGURES_DIR   = REPO_ROOT / "outputs" / "figures"
TABLES_DIR    = REPO_ROOT / "outputs" / "tables"

CONTRACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

print(f"REPO_ROOT:     {REPO_ROOT}")
print(f"CONTRACTS_DIR: {CONTRACTS_DIR}")

REPO_ROOT:     C:\Users\dunla\repos\data-center-flexibility-resource-adequacy
CONTRACTS_DIR: C:\Users\dunla\repos\data-center-flexibility-resource-adequacy\outputs\contracts


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-2: LOAD CONTRACTS FROM NOTEBOOK 01
# ══════════════════════════════════════════════════════════════════════════════

with open(CONTRACTS_DIR / "stress_correlation_results.json") as f:
    stress_results = json.load(f)

per_hour_avail = pd.read_parquet(CONTRACTS_DIR / "per_hour_destination_availability.parquet")

with open(CONTRACTS_DIR / "workload_parameters.json") as f:
    workload = json.load(f)

# S3 flows through from workload contract — single source of truth for S3=0.90
# across the cascade. Notebook 01 writes it; notebook 02 reads it; notebook 03
# consumes it via the cascade_parameters.json export written in Cell 5-1 below.
S3_FROM_CONTRACT = workload["s3_parameterization"]

print(f"stress_results top-level keys: {list(stress_results.keys())}")
print(f"per_hour_avail: {per_hour_avail.shape} ({len(per_hour_avail.columns)} cols)")
print(f"workload: S3 = {S3_FROM_CONTRACT}")

stress_results top-level keys: ['metadata', 'headline', 'empirical_destination_lmps', 'per_zone', 'yearly']
per_hour_avail: (200, 21) (21 cols)
workload: S3 = {'value': 0.9, 'justification': 'P99 drain times across two independent datasets range from 4.5 to 12.1 seconds at 60 tok/s, approximately two orders of magnitude below PJM 10-minute dispatch window. S3 = 0.90 reserves 10% of load for long-running or batched requests.', 'pjm_dispatch_window_seconds': 600}


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0-3: CONTRACT SANITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════
# Fail loudly if upstream contracts are malformed, stale, or missing expected
# fields. These checks guard against silent drift between notebook 01 outputs
# and notebook 02 inputs.
# ══════════════════════════════════════════════════════════════════════════════

# ─── Parquet structural checks ───────────────────────────────────────────────
assert per_hour_avail.shape == (200, 21), (
    f"Unexpected parquet shape: {per_hour_avail.shape}, expected (200, 21)"
)
assert "timestamp" in per_hour_avail.columns, "timestamp column missing"
assert "pjm_co_stressed_mw" in per_hour_avail.columns, "pjm_co_stressed_mw column missing"

dest_cols = [c for c in per_hour_avail.columns if c not in ("timestamp", "pjm_co_stressed_mw")]
assert len(dest_cols) == 19, f"Expected 19 destination columns, got {len(dest_cols)}"

# ─── Cell 4-2 fix verification ───────────────────────────────────────────────
# Pre-fix, notebook 01 filtered dest_zones by 'PJM' string, which never matched
# (all dest zones are cross-BA: CAISO/ERCOT/MISO/NYISO), leaving pjm_co_stressed_mw
# all zeros. Post-fix should have ~30 non-zero hours averaging several GW.
n_nonzero = int((per_hour_avail["pjm_co_stressed_mw"] > 0).sum())
assert n_nonzero > 0, "pjm_co_stressed_mw is all zeros — Cell 4-2 fix did not persist"
assert n_nonzero >= 20, f"Only {n_nonzero} non-zero PJM co-stress hours; expected at least 20"

# ─── Workload contract ───────────────────────────────────────────────────────
# S3 is stored as a nested dict in workload_parameters.json: the numeric value
# travels with its justification and the PJM dispatch window reference. Unpack
# the scalar here so the rest of the notebook has a single S3_VALUE to use.
assert isinstance(S3_FROM_CONTRACT, dict), (
    f"Expected s3_parameterization to be a dict, got {type(S3_FROM_CONTRACT).__name__}"
)
assert "value" in S3_FROM_CONTRACT, "s3_parameterization missing 'value' field"
S3_VALUE = float(S3_FROM_CONTRACT["value"])
assert S3_VALUE == 0.90, f"S3 value = {S3_VALUE}, expected 0.90"

# ─── Summary ─────────────────────────────────────────────────────────────────
pjm_mean = per_hour_avail.loc[per_hour_avail["pjm_co_stressed_mw"] > 0, "pjm_co_stressed_mw"].mean()
print(f"✓ Contract sanity checks passed")
print(f"  Stress hours:                   {len(per_hour_avail)}")
print(f"  Destination zones (cross-BA):   {len(dest_cols)}")
print(f"  PJM co-stress non-zero hours:   {n_nonzero} / {len(per_hour_avail)}")
print(f"  PJM co-stress mean (non-zero):  {pjm_mean:,.0f} MW")
print(f"  S3 (from workload contract):    {S3_VALUE}")

✓ Contract sanity checks passed
  Stress hours:                   200
  Destination zones (cross-BA):   19
  PJM co-stress non-zero hours:   30 / 200
  PJM co-stress mean (non-zero):  4,631 MW
  S3 (from workload contract):    0.9


## Part 1: Cascade Framework (§2)

- **1-1** Ten-parameter definitions, ranges, and `FLEX_FRAC`
- **1-2** Static cascade product (conservative / central / optimistic)
- **1-3** Commitment depth formula (Colangelo-corrected DVFS-on-S1-envelope)

Decomposes spatial migration feasibility into ten multiplicative parameters: three source-side (S1, S2, S3), five destination-side (D1–D5), and two execution (E1, E2). The **effective spatial fraction** is the product of all ten, representing the share of facility load that can be shifted off the source node during a capacity market dispatch event. The **commitment depth** adds DVFS flexibility on the shiftable-compute residual within the S1 envelope (Colangelo et al. 2025, corrected formula applied to the S1 envelope rather than 1 − spatial).

Parameter values and ranges reconciled against NE manuscript Methods. S3 central flows from `workload_parameters.json` (notebook 01 output) rather than being hardcoded, so any future reparameterization of S3 based on updated drain-time data propagates automatically through the cascade.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1-1: TEN-PARAMETER CASCADE DEFINITIONS AND RANGES
# ══════════════════════════════════════════════════════════════════════════════
# Effective Spatial Fraction = S1 × S2 × S3 × D1 × D2 × D3 × D4 × D5 × E1 × E2
#
# Each parameter is an independent filter on the spatial migration pathway.
# Grounding tags indicate source quality:
#   DATA                : empirically grounded
#   STRUCTURAL          : definitional (e.g. inference share by design)
#   ESTIMATED-MODERATE  : published literature with disagreement
#   ESTIMATED-WEAK      : no direct measurement; regulatory/modeling judgment
#
# Reconciled values per NE manuscript Methods. See notebook 02 README for the
# reconciliation history (v18 scalars were updated in place but ranges drifted;
# this cell is the single clean source of truth).
# ══════════════════════════════════════════════════════════════════════════════

# ─── Source-side ─────────────────────────────────────────────────────────────
CASCADE_S1 = 0.70           # [STRUCTURAL] Workload candidacy (inference-dominant scenario).
                            #   Fixed at 0.70 within the inference-dominant framing; not sampled.

CASCADE_S2 = 0.70           # [ESTIMATED-WEAK] Data locality.
                            #   Central = 0.70 reflects a conservative "regulatory floor" framing
                            #   anchored on data residency requirements for cross-region inference
                            #   routing (same-country, same-jurisdiction). Range [0.60, 0.95]
                            #   accommodates stricter or looser residency regimes. No published
                            #   shiftability measurement exists; this is the primary modeling
                            #   vulnerability flagged in §2.

CASCADE_S3 = S3_VALUE       # [DATA] Operational feasibility given advance notice.
                            #   Sourced from outputs/contracts/workload_parameters.json (notebook 01).
                            #   Central 0.90 reserves 10% of load for long-running or batched
                            #   requests whose drain time exceeds the PJM 10-minute dispatch window.
                            #   Grounded in DynamoLLM and BurstGPT P99 drain-time analysis
                            #   (see workload contract justification).

# ─── Destination-side ────────────────────────────────────────────────────────
CASCADE_D1 = 0.99           # [DATA] Destination availability.
                            #   Pillar 2 empirical: 198/200 ComEd stress hours had cross-BA
                            #   destinations available (99.0%). See notebook 01 Part 1.

CASCADE_D2 = 0.33           # [ESTIMATED-WEAK] Utilization headroom = 1 - util.
                            #   Central util ~0.67 per SemiAnalysis/MIT GPU measurements.
                            #   Range [0.20, 0.50] reflects GPU utilization 50-80%.
                            #   Highest variance contribution (~44%) per Methods variance
                            #   decomposition; addressed directly in Part 4 sensitivity surface.

CASCADE_D3 = 0.88           # [ESTIMATED-MODERATE] Hardware compatibility (CUDA backward compat,
                            #   TensorRT). compatible_fraction_reference Layer 1: 0.80-0.92.
                            #   Excludes pre-staging, which is now D5.

CASCADE_D4 = 0.50           # [ESTIMATED-MODERATE] Inference workload share at destination.
                            #   compatible_fraction_reference Layer 2: 0.33-0.67.
                            #   Deloitte (Nov 2025): ~50% in 2025; McKinsey: 30-40% by 2030.

CASCADE_D5 = 0.65           # [ESTIMATED-WEAK] Operational readiness / pre-staging.
                            #   compatible_fraction_reference Layer 3: 0.50-0.80.
                            #   Weakest layer per Pillar 1 TGV. Hyperscaler own-fleet 0.60-0.80;
                            #   merchant floor 0.50. Central = 0.65 from reference central scenario.

# ─── Execution ───────────────────────────────────────────────────────────────
CASCADE_E1 = 0.95           # [ESTIMATED] Routing completion.
                            #   Conservative SLA envelope below measured cloud load-balancer
                            #   failure rates (~0.999 steady-state), to accommodate DNS propagation,
                            #   health-check latency, and cross-BA routing variability not captured
                            #   in published SLAs. Range widened from Bartlett v18's tight (0.995,
                            #   0.999) to reflect uncertainty in conservative framing. Variance
                            #   contribution <5% per Methods; result is numerically insensitive.

CASCADE_E2 = 0.98           # [ESTIMATED] Bandwidth adequacy.
                            #   Conservative SLA envelope below hyperscaler WAN headroom implied
                            #   by the orders-of-magnitude argument in Methods §E2. Deliberate
                            #   buffer for unmodeled congestion events. Variance contribution
                            #   <5% per Methods; result is numerically insensitive.

# ─── DVFS flexibility floor ──────────────────────────────────────────────────
FLEX_FRAC = 0.25            # [DATA] DVFS flexibility on shiftable-compute residual.
                            #   Colangelo et al. 2025 measured ~25% power reduction at the GPU
                            #   cluster (not facility meter). Applied only within the S1 envelope
                            #   in the commitment depth formula (Cell 1-3).

# ─── Parameter ranges (conservative, central, optimistic) ────────────────────
# Single source of truth for all downstream sensitivity and variance-decomp work.
# CENTRAL column must equal the scalar values above — enforced by assertion below.
CASCADE_RANGES = {
    'S1': (0.70,  0.70,  0.70),     # Fixed within inference-dominant scenario
    'S2': (0.60,  0.70,  0.95),     # Reconciled: central moved 0.80→0.70, endpoints unchanged
    'S3': (0.85,  0.90,  0.95),     # Matches workload-contract central; endpoints unchanged
    'D1': (0.945, 0.99,  0.995),    # Empirical-forward range for climate/reflexivity
    'D2': (0.20,  0.33,  0.50),     # Narrowed per Methods: GPU util 50-80% → headroom 0.20-0.50
    'D3': (0.80,  0.88,  0.92),     # HW compat Layer 1 (excludes pre-staging)
    'D4': (0.33,  0.50,  0.67),     # Inference share Layer 2
    'D5': (0.50,  0.65,  0.80),     # Pre-staging Layer 3
    'E1': (0.90,  0.95,  0.99),     # Reconciled: widened to contain new central 0.95
    'E2': (0.95,  0.98,  0.995),    # Reconciled: widened to contain new central 0.98
}

# ─── Consistency check: scalars must match range centrals ────────────────────
_scalars = {
    'S1': CASCADE_S1, 'S2': CASCADE_S2, 'S3': CASCADE_S3,
    'D1': CASCADE_D1, 'D2': CASCADE_D2, 'D3': CASCADE_D3,
    'D4': CASCADE_D4, 'D5': CASCADE_D5,
    'E1': CASCADE_E1, 'E2': CASCADE_E2,
}
for _k, _v in _scalars.items():
    _central = CASCADE_RANGES[_k][1]
    assert abs(_v - _central) < 1e-9, (
        f"{_k} scalar ({_v}) differs from CASCADE_RANGES central ({_central}). "
        f"Fix one or the other — they must agree."
    )

print("TEN-PARAMETER CASCADE — central values")
print("─" * 60)
print(f"  Source:       S1={CASCADE_S1:.2f}  S2={CASCADE_S2:.2f}  S3={CASCADE_S3:.2f}")
print(f"  Destination:  D1={CASCADE_D1:.3f} D2={CASCADE_D2:.2f}  D3={CASCADE_D3:.2f}")
print(f"                D4={CASCADE_D4:.2f}  D5={CASCADE_D5:.2f}")
print(f"  Execution:    E1={CASCADE_E1:.2f}  E2={CASCADE_E2:.2f}")
print(f"  DVFS:         FLEX_FRAC={FLEX_FRAC:.2f}")
print(f"✓ Scalar/range consistency check passed")

TEN-PARAMETER CASCADE — central values
────────────────────────────────────────────────────────────
  Source:       S1=0.70  S2=0.70  S3=0.90
  Destination:  D1=0.990 D2=0.33  D3=0.88
                D4=0.50  D5=0.65
  Execution:    E1=0.95  E2=0.98
  DVFS:         FLEX_FRAC=0.25
✓ Scalar/range consistency check passed


The parameter definitions and ranges above are the single source of truth for every downstream cascade computation in this notebook. The next two cells compute **point estimates** at the range endpoints (Cell 1-2) and the **Colangelo-corrected commitment depth** at the central point (Cell 1-3). The full uncertainty propagation — unconditional Monte Carlo and variance attribution — lives in Part 2.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1-2: STATIC CASCADE PRODUCT (conservative / central / optimistic)
# ══════════════════════════════════════════════════════════════════════════════
# Point estimates at the three range endpoints, computed from CASCADE_RANGES.
# The central value is the headline number cited in §2 and Methods.
# ══════════════════════════════════════════════════════════════════════════════

def _cascade_product(vals):
    """Product of all ten cascade parameters given a {name: value} dict."""
    return (vals['S1'] * vals['S2'] * vals['S3'] *
            vals['D1'] * vals['D2'] * vals['D3'] *
            vals['D4'] * vals['D5'] *
            vals['E1'] * vals['E2'])

_con_vals     = {k: v[0] for k, v in CASCADE_RANGES.items()}
_central_vals = {k: v[1] for k, v in CASCADE_RANGES.items()}
_opt_vals     = {k: v[2] for k, v in CASCADE_RANGES.items()}

CASCADE_CONSERVATIVE   = _cascade_product(_con_vals)
EFFECTIVE_SPATIAL_FRAC = _cascade_product(_central_vals)
CASCADE_OPTIMISTIC     = _cascade_product(_opt_vals)

print("CASCADE PRODUCT")
print("─" * 60)
print(f"  Conservative:  {CASCADE_CONSERVATIVE:.4f}")
print(f"  Central:       {EFFECTIVE_SPATIAL_FRAC:.4f}")
print(f"  Optimistic:    {CASCADE_OPTIMISTIC:.4f}")

# ─── Regression check against reconciled Bartlett v18 target ─────────────────
assert abs(EFFECTIVE_SPATIAL_FRAC - 0.0384) < 0.001, (
    f"Central cascade product {EFFECTIVE_SPATIAL_FRAC:.4f} does not match "
    f"reconciled Bartlett v18 target 0.0384. Investigate parameter drift."
)
print(f"✓ Regression check: central product matches reconciled Bartlett v18 (0.0384)")

CASCADE PRODUCT
────────────────────────────────────────────────────────────
  Conservative:  0.0076
  Central:       0.0384
  Optimistic:    0.1527
✓ Regression check: central product matches reconciled Bartlett v18 (0.0384)


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1-3: COMMITMENT DEPTH (Colangelo-corrected formula)
# ══════════════════════════════════════════════════════════════════════════════
# Commitment depth = spatial_fraction + FLEX_FRAC × (S1 - spatial_fraction)
#
# The DVFS multiplier applies to the shiftable-compute residual WITHIN the S1
# envelope, not to (1 - spatial). Colangelo et al. 2025 measured the 25% power
# reduction at the GPU cluster, not the facility meter. Applying DVFS to
# (1 - spatial) would include non-compute facility load (cooling, power
# conditioning) in the DVFS base, overstating the flexibility floor.
#
# The DVFS-only floor (no spatial migration at all) is FLEX_FRAC × S1 = 17.5%.
# This is the minimum commitment depth achievable through compute flexibility
# alone, without any cross-BA migration.
# ══════════════════════════════════════════════════════════════════════════════

COMMITMENT_DEPTH_BASELINE = (
    EFFECTIVE_SPATIAL_FRAC + FLEX_FRAC * (CASCADE_S1 - EFFECTIVE_SPATIAL_FRAC)
)

DVFS_ONLY_FLOOR = FLEX_FRAC * CASCADE_S1  # 0.25 × 0.70 = 17.5%

# Also compute commitment depth at conservative and optimistic endpoints
_depth_con = CASCADE_CONSERVATIVE + FLEX_FRAC * (CASCADE_S1 - CASCADE_CONSERVATIVE)
_depth_opt = CASCADE_OPTIMISTIC   + FLEX_FRAC * (CASCADE_S1 - CASCADE_OPTIMISTIC)

print("COMMITMENT DEPTH")
print("─" * 60)
print(f"  DVFS-only floor:       {DVFS_ONLY_FLOOR:>6.1%}  (no spatial migration)")
print(f"  Conservative cascade:  {_depth_con:>6.1%}  (spatial {CASCADE_CONSERVATIVE:.4f} + DVFS residual)")
print(f"  Central cascade:       {COMMITMENT_DEPTH_BASELINE:>6.1%}  (spatial {EFFECTIVE_SPATIAL_FRAC:.4f} + DVFS residual)")
print(f"  Optimistic cascade:    {_depth_opt:>6.1%}  (spatial {CASCADE_OPTIMISTIC:.4f} + DVFS residual)")

# ─── Regression check against reconciled Bartlett v18 target ─────────────────
assert abs(COMMITMENT_DEPTH_BASELINE - 0.204) < 0.002, (
    f"Central commitment depth {COMMITMENT_DEPTH_BASELINE:.4f} does not match "
    f"reconciled Bartlett v18 target 0.204 (20.4%). Investigate."
)
print(f"✓ Regression check: central commitment depth matches reconciled target (20.4%)")

COMMITMENT DEPTH
────────────────────────────────────────────────────────────
  DVFS-only floor:        17.5%  (no spatial migration)
  Conservative cascade:   18.1%  (spatial 0.0076 + DVFS residual)
  Central cascade:        20.4%  (spatial 0.0384 + DVFS residual)
  Optimistic cascade:     29.0%  (spatial 0.1527 + DVFS residual)
✓ Regression check: central commitment depth matches reconciled target (20.4%)


## Part 2: Unconditional Variance Decomposition (§2, Methods)

- **2-1** Monte Carlo over all ten cascade parameters (N=50,000), variance attribution via η² correlation ratios
- **2-2** Scenario summary table (conservative / central / optimistic) and one-at-a-time parameter sensitivity

Propagates parameter uncertainty through the cascade product **without** conditioning on empirical grid state — D1 is sampled from its prior range rather than read from the per-hour parquet. This produces the variance shares cited in Methods as evidence that D2, D4, D5, and S2 are the dominant uncertainty drivers and therefore the empirical investment priorities for tightening the cascade estimate. The conditional (empirical) Monte Carlo in Part 3 is the complement: it reads the real per-hour destination availability from notebook 01's stress correlation contract and computes commitment depth at the actually-observed grid states.

S1 is held fixed at 0.70 (scenario-defining scalar, not drawn); all nine other parameters are sampled independently. D1 uses a triangular distribution peaked at the empirical central value (0.99); all other sampled parameters use uniform distributions over their `CASCADE_RANGES` endpoints.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2-1: CASCADE MONTE CARLO + VARIANCE DECOMPOSITION
# ══════════════════════════════════════════════════════════════════════════════
# Monte Carlo sensitivity analysis for the ten-parameter cascade.
# Produces:
#   1. Distribution of cascade product and commitment depth
#   2. Variance attribution (η² correlation ratios) identifying which
#      parameters drive cascade uncertainty under independent sampling
#
# Sampling:
#   S1: held fixed (scenario-definer, not drawn)
#   D1: triangular(con, central, opt) — concentrates mass at empirical anchor
#   All others: uniform(con, opt) over CASCADE_RANGES endpoints
#
# Variance attribution is computed on the cascade product. Because commitment
# depth is an affine function of the product at fixed S1
# (commit = product + FLEX_FRAC × (S1 - product)), the variance SHARES are
# identical whether computed on product or commitment.
#
# INPUTS:  CASCADE_S1, CASCADE_RANGES, FLEX_FRAC (from Part 1)
# OUTPUTS: CASCADE_MC_PRODUCTS, CASCADE_MC_COMMITMENT, CASCADE_ETA_SQ
# ══════════════════════════════════════════════════════════════════════════════

np.random.seed(429)
N = 50_000

S1_FIXED = CASCADE_S1  # held fixed; not sampled

# ─── Draw samples from CASCADE_RANGES (single source of truth) ───────────────
# All parameters except S1 and D1 use uniform(con, opt).
# D1 uses triangular peaked at central to reflect its empirical grounding.
_sampled_params = ['S2', 'S3', 'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2']

samples = {}
for _p in _sampled_params:
    _con, _central, _opt = CASCADE_RANGES[_p]
    if _p == 'D1':
        samples[_p] = np.random.triangular(_con, _central, _opt, N)
    else:
        samples[_p] = np.random.uniform(_con, _opt, N)

# ─── Cascade product and commitment depth for each draw ──────────────────────
CASCADE_MC_PRODUCTS = (
    S1_FIXED *
    samples['S2'] * samples['S3'] *
    samples['D1'] * samples['D2'] * samples['D3'] *
    samples['D4'] * samples['D5'] *
    samples['E1'] * samples['E2']
)

CASCADE_MC_COMMITMENT = (
    CASCADE_MC_PRODUCTS + FLEX_FRAC * (S1_FIXED - CASCADE_MC_PRODUCTS)
)

# ─── Summary distribution ────────────────────────────────────────────────────
_p = CASCADE_MC_PRODUCTS
_c = CASCADE_MC_COMMITMENT

print(f"CASCADE MONTE CARLO (N={N:,}, 9 sampled parameters + S1 fixed)")
print("=" * 74)
print(f"  Cascade product:   mean={np.mean(_p):.4f}  std={np.std(_p):.4f}  "
      f"[P5={np.percentile(_p,5):.4f}, P95={np.percentile(_p,95):.4f}]")
print(f"  Commitment depth:  mean={np.mean(_c):.1%}  std={np.std(_c):.4f}  "
      f"[P5={np.percentile(_c,5):.1%}, P95={np.percentile(_c,95):.1%}]")

# ─── Variance attribution (η² correlation ratios) ────────────────────────────
# For each sampled parameter:
#   1. Digitize into 20 quantile bins
#   2. Compute between-group variance of the cascade product across bins
#   3. Divide by total variance of the cascade product
# Result is the fraction of product variance explained by that parameter alone,
# approximating the first-order Sobol index.
# ─────────────────────────────────────────────────────────────────────────────

_total_var = np.var(CASCADE_MC_PRODUCTS)
_mean_p = np.mean(CASCADE_MC_PRODUCTS)
CASCADE_ETA_SQ = {}

for _param, _vals in samples.items():
    _bin_edges = np.percentile(_vals, np.linspace(0, 100, 21))
    _bins = np.digitize(_vals, _bin_edges)
    _group_means = []
    _group_counts = []
    for _b in range(1, 21):
        _mask = (_bins == _b)
        _n_b = int(_mask.sum())
        if _n_b > 0:
            _group_means.append(CASCADE_MC_PRODUCTS[_mask].mean())
            _group_counts.append(_n_b)
    _group_means = np.array(_group_means)
    _group_counts = np.array(_group_counts)
    _between_var = np.sum(_group_counts * (_group_means - _mean_p)**2) / N
    CASCADE_ETA_SQ[_param] = _between_var / _total_var

# Grounding tags (consistent with Cell 1-1 comments)
_grounding = {
    'S2': 'ESTIMATED-WEAK',
    'S3': 'DATA',
    'D1': 'DATA',
    'D2': 'ESTIMATED-WEAK',
    'D3': 'ESTIMATED-MODERATE',
    'D4': 'ESTIMATED-MODERATE',
    'D5': 'ESTIMATED-WEAK',
    'E1': 'ESTIMATED',
    'E2': 'DATA',
}

_eta_sum = sum(CASCADE_ETA_SQ.values())

print(f"\nVARIANCE ATTRIBUTION (η² correlation ratios)")
print("=" * 74)
print(f"  {'Param':<6} | {'η²':>8} | {'% of Σ η²':>10} | {'Grounding':<20}")
print(f"  {'─' * 68}")
for _param, _eta in sorted(CASCADE_ETA_SQ.items(), key=lambda x: -x[1]):
    _pct = 100 * _eta / _eta_sum
    print(f"  {_param:<6} | {_eta:>8.4f} | {_pct:>9.1f}% | {_grounding[_param]:<20}")
print(f"  {'─' * 68}")
print(f"  Σ η² = {_eta_sum:.4f}  (≈1.0 confirms negligible parameter interactions)")

# ─── Compare against paper Methods cited shares ──────────────────────────────
# Paper §29 cites D2=44%, D4=28%, D5=13%, and S2≈13% (§40).
# If reconciled E1/E2 ranges move these meaningfully, the paper needs updating.
print(f"\nPAPER METHODS COMPARISON")
print("=" * 74)
_paper_shares = {'D2': 0.44, 'D4': 0.28, 'D5': 0.13, 'S2': 0.13}
print(f"  {'Param':<6} | {'Paper cite':>10} | {'Computed':>10} | {'Δ (pp)':>8}")
print(f"  {'─' * 44}")
for _param, _paper_pct in _paper_shares.items():
    _computed_pct = CASCADE_ETA_SQ[_param] / _eta_sum
    _delta_pp = 100 * (_computed_pct - _paper_pct)
    _flag = "  ← check" if abs(_delta_pp) > 2.0 else ""
    print(f"  {_param:<6} | {_paper_pct:>9.1%} | {_computed_pct:>9.1%} | {_delta_pp:>+6.1f}pp{_flag}")

CASCADE MONTE CARLO (N=50,000, 9 sampled parameters + S1 fixed)
  Cascade product:   mean=0.0429  std=0.0163  [P5=0.0208, P95=0.0734]
  Commitment depth:  mean=20.7%  std=0.0122  [P5=19.1%, P95=23.0%]

VARIANCE ATTRIBUTION (η² correlation ratios)
  Param  |       η² |  % of Σ η² | Grounding           
  ────────────────────────────────────────────────────────────────────
  D2     |   0.4205 |      43.9% | ESTIMATED-WEAK      
  D4     |   0.2668 |      27.9% | ESTIMATED-MODERATE  
  D5     |   0.1238 |      12.9% | ESTIMATED-WEAK      
  S2     |   0.1197 |      12.5% | ESTIMATED-WEAK      
  D3     |   0.0096 |       1.0% | ESTIMATED-MODERATE  
  S3     |   0.0083 |       0.9% | DATA                
  E1     |   0.0052 |       0.5% | ESTIMATED           
  E2     |   0.0017 |       0.2% | DATA                
  D1     |   0.0012 |       0.1% | DATA                
  ────────────────────────────────────────────────────────────────────
  Σ η² = 0.9569  (≈1.0 confirms negligible paramete

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2-2: SCENARIO SUMMARY AND ONE-AT-A-TIME SENSITIVITY
# ══════════════════════════════════════════════════════════════════════════════
# Two complementary views of parameter uncertainty:
#
# 1. SCENARIO SUMMARY — point estimates at the three range corners
#    (all parameters simultaneously at conservative / central / optimistic).
#    These are JOINT WORST/BEST CASE bounds, not probabilistic bounds. The
#    meaningful uncertainty envelope is the MC P5/P95 from Cell 2-1.
#
# 2. ONE-AT-A-TIME SENSITIVITY — marginal impact of each parameter, sweeping
#    low → high while holding all others at their central values. Complements
#    Cell 2-1's joint variance decomposition: variance shares answer
#    "which parameter drives uncertainty when everything moves?", tornado
#    answers "which parameter moves the output most when held in isolation?"
# ══════════════════════════════════════════════════════════════════════════════

# ─── Scenario summary table ──────────────────────────────────────────────────
_central_depth = (
    EFFECTIVE_SPATIAL_FRAC + FLEX_FRAC * (CASCADE_S1 - EFFECTIVE_SPATIAL_FRAC)
)
_con_depth = (
    CASCADE_CONSERVATIVE + FLEX_FRAC * (CASCADE_S1 - CASCADE_CONSERVATIVE)
)
_opt_depth = (
    CASCADE_OPTIMISTIC + FLEX_FRAC * (CASCADE_S1 - CASCADE_OPTIMISTIC)
)

CASCADE_SCENARIOS = {
    'Conservative': {'cascade': CASCADE_CONSERVATIVE, 'depth': _con_depth},
    'Central':      {'cascade': EFFECTIVE_SPATIAL_FRAC, 'depth': _central_depth},
    'Optimistic':   {'cascade': CASCADE_OPTIMISTIC,    'depth': _opt_depth},
}

print("CASCADE SCENARIO SUMMARY")
print("=" * 74)
print(f"  {'Scenario':<14} | {'Cascade':>9} | {'Depth':>7} | {'Firm':>6}")
print(f"  {'':14} | {'Product':>9} | {'':>7} | {'Share':>6}")
print(f"  {'─' * 44}")
for _label, _s in CASCADE_SCENARIOS.items():
    _firm = 1 - _s['depth']
    print(f"  {_label:<14} | {_s['cascade']:>9.4f} | {_s['depth']:>6.1%} | {_firm:>5.1%}")
print(f"  {'─' * 44}")
print(f"  {'DVFS-only floor':<14} | {'—':>9} | {DVFS_ONLY_FLOOR:>6.1%} | {1-DVFS_ONLY_FLOOR:>5.1%}")
print("\n  Note: conservative/optimistic are JOINT WORST/BEST corner estimates,")
print("  not probabilistic bounds. Use Cell 2-1 MC P5/P95 for uncertainty envelope.")

# ─── Monte Carlo uncertainty envelope (from Cell 2-1) ────────────────────────
print(f"\n  MC-derived uncertainty envelope (N={N:,}, P5–P95):")
print(f"    Cascade product:   "
      f"[{np.percentile(CASCADE_MC_PRODUCTS,5):.4f}, "
      f"{np.percentile(CASCADE_MC_PRODUCTS,95):.4f}]")
print(f"    Commitment depth:  "
      f"[{np.percentile(CASCADE_MC_COMMITMENT,5):.1%}, "
      f"{np.percentile(CASCADE_MC_COMMITMENT,95):.1%}]")

# ─── One-at-a-time parameter sensitivity ─────────────────────────────────────
# Hold nine parameters at central, sweep the tenth low→high, record the
# cascade product and commitment depth at each endpoint. Sorted descending
# by absolute impact on commitment depth (tornado ordering).
# ─────────────────────────────────────────────────────────────────────────────

_central_vals = {k: v[1] for k, v in CASCADE_RANGES.items()}

def _cascade_from_dict(d):
    return (d['S1'] * d['S2'] * d['S3'] *
            d['D1'] * d['D2'] * d['D3'] *
            d['D4'] * d['D5'] *
            d['E1'] * d['E2'])

def _commit_from_cascade(c):
    return c + FLEX_FRAC * (CASCADE_S1 - c)

_oat_rows = []
for _param in CASCADE_RANGES.keys():
    if _param == 'S1':
        continue  # S1 is fixed; no sweep
    _con, _central, _opt = CASCADE_RANGES[_param]

    _vals_lo = dict(_central_vals); _vals_lo[_param] = _con
    _vals_hi = dict(_central_vals); _vals_hi[_param] = _opt

    _casc_lo = _cascade_from_dict(_vals_lo)
    _casc_hi = _cascade_from_dict(_vals_hi)
    _depth_lo = _commit_from_cascade(_casc_lo)
    _depth_hi = _commit_from_cascade(_casc_hi)

    _oat_rows.append({
        'param': _param,
        'lo': _con, 'hi': _opt,
        'depth_lo': _depth_lo, 'depth_hi': _depth_hi,
        'abs_impact_pp': 100 * abs(_depth_hi - _depth_lo),
    })

_oat_rows.sort(key=lambda r: -r['abs_impact_pp'])

print(f"\nONE-AT-A-TIME PARAMETER SENSITIVITY (sorted by |Δ depth|)")
print("=" * 74)
print(f"  {'Param':<6} | {'Lo':>6} | {'Hi':>6} | {'Depth Lo':>9} | "
      f"{'Depth Hi':>9} | {'|Δ|':>7}")
print(f"  {'─' * 62}")
for _r in _oat_rows:
    print(f"  {_r['param']:<6} | {_r['lo']:>6.3f} | {_r['hi']:>6.3f} | "
          f"{_r['depth_lo']:>8.1%} | {_r['depth_hi']:>8.1%} | "
          f"{_r['abs_impact_pp']:>5.1f}pp")

CASCADE SCENARIO SUMMARY
  Scenario       |   Cascade |   Depth |   Firm
                 |   Product |         |  Share
  ────────────────────────────────────────────
  Conservative   |    0.0076 |  18.1% | 81.9%
  Central        |    0.0384 |  20.4% | 79.6%
  Optimistic     |    0.1527 |  29.0% | 71.0%
  ────────────────────────────────────────────
  DVFS-only floor |         — |  17.5% | 82.5%

  Note: conservative/optimistic are JOINT WORST/BEST corner estimates,
  not probabilistic bounds. Use Cell 2-1 MC P5/P95 for uncertainty envelope.

  MC-derived uncertainty envelope (N=50,000, P5–P95):
    Cascade product:   [0.0208, 0.0734]
    Commitment depth:  [19.1%, 23.0%]

ONE-AT-A-TIME PARAMETER SENSITIVITY (sorted by |Δ depth|)
  Param  |     Lo |     Hi |  Depth Lo |  Depth Hi |     |Δ|
  ──────────────────────────────────────────────────────────────
  D2     |  0.200 |  0.500 |    19.2% |    21.9% |   2.6pp
  D4     |  0.330 |  0.670 |    19.4% |    21.4% |   2.0pp
  S2     |  0.6